In [3]:
import time
import csv
from urllib.parse import urlparse
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By

START_URL = "https://www.thailand-property.com/properties-for-sale?exact_bed=false"
OUTPUT_FILE = "thailand_property_urls.csv"
TIMEOUT = 40
MAX_PAGES = 20

def wait_ready(driver):
    t0 = time.time()
    while time.time() - t0 < TIMEOUT:
        if driver.execute_script("return document.readyState") == "complete":
            return True
        time.sleep(0.2)
    return False

def check_server_error(driver):
    page_source = driver.page_source.lower()
    if "server error in '/' application" in page_source or "timeout expired" in page_source:
        return True
    return False

def extract_links(driver, base):
    js = """
    return [...new Set(
        Array.from(document.querySelectorAll("a.hj-listing-snippet"))
        .map(a => a.getAttribute('href'))
        .filter(h => h && h.includes('/ads/'))
    )];
    """
    hrefs = driver.execute_script(js)

    out = []
    for h in hrefs:
        out.append(base + h if h.startswith("/") else h)

    return list(dict.fromkeys(out))

def scroll(driver):
    last_h = 0
    for _ in range(10):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.0)
        h = driver.execute_script("return document.body.scrollHeight")
        if h == last_h:
            break
        last_h = h

def click_next(driver):
    try:
        next_btn = driver.find_element(By.XPATH, "//a[@rel='next']")
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", next_btn)
        time.sleep(1)
        driver.execute_script("arguments[0].click();", next_btn)
        return True
    except Exception:
        return False

def main():
    opt = uc.ChromeOptions()
    opt.add_argument("--no-sandbox")
    opt.add_argument("--disable-dev-shm-usage")
    opt.add_argument("--disable-gpu")
    opt.add_argument("--disable-notifications")
    opt.add_argument("--disable-blink-features=AutomationControlled")

    driver = uc.Chrome(options=opt, version_main=145)
    base = f"{urlparse(START_URL).scheme}://{urlparse(START_URL).netloc}"

    print(f"Loading initial URL: {START_URL}")
    driver.get(START_URL)
    wait_ready(driver)
    time.sleep(3)

    page = 1
    urls = set()
    retries = 0
    max_retries = 3

    while page <= MAX_PAGES:
        if check_server_error(driver):
            print(f"[!] Server error detected on page {page}. Retrying in 10 seconds... ({retries+1}/{max_retries})")
            time.sleep(10)
            driver.refresh()
            wait_ready(driver)
            time.sleep(5)
            retries += 1
            if retries > max_retries:
                print("[!] Max retries reached. Stopping scraping.")
                break
            continue

        retries = 0

        print(f"[Page {page}/{MAX_PAGES}] Scrolling and collecting links...")
        scroll(driver)
        found = extract_links(driver, base)

        if found:
            urls.update(found)
            print(f"  -> Found {len(found)} listings (total: {len(urls)})")
        else:
            print("  -> Found 0 listings")

        if page >= MAX_PAGES:
            print(f"  -> Reached max pages ({MAX_PAGES}). Stopping.")
            break

        if not click_next(driver):
            print("  -> No next button found or last page reached")
            break

        print(f"[Page {page}] Navigated to next page")
        time.sleep(4)
        wait_ready(driver)
        page += 1

    driver.quit()

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["ListingURL"])
        for u in sorted(urls):
            writer.writerow([u])

    print(f"\nDone. Total {len(urls)} listings saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Loading initial URL: https://www.thailand-property.com/properties-for-sale?exact_bed=false
[Page 1/20] Scrolling and collecting links...
  -> Found 25 listings (total: 25)
[Page 1] Navigated to next page
[Page 2/20] Scrolling and collecting links...
  -> Found 25 listings (total: 50)
[Page 2] Navigated to next page
[Page 3/20] Scrolling and collecting links...
  -> Found 25 listings (total: 75)
[Page 3] Navigated to next page
[Page 4/20] Scrolling and collecting links...
  -> Found 25 listings (total: 100)
[Page 4] Navigated to next page
[Page 5/20] Scrolling and collecting links...
  -> Found 25 listings (total: 125)
[Page 5] Navigated to next page
[Page 6/20] Scrolling and collecting links...
  -> Found 25 listings (total: 150)
[Page 6] Navigated to next page
[Page 7/20] Scrolling and collecting links...
  -> Found 25 listings (total: 175)
[Page 7] Navigated to next page
[Page 8/20] Scrolling and collecting links...
  -> Found 25 listings (total: 200)
[Page 8] Navigated to next page


In [2]:
import os
import csv
import time
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By

BASE_DIR = '/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/thailand-property'
INPUT_CSV = os.path.join(BASE_DIR, 'thailand_property_urls.csv')
OUTPUT_CSV = os.path.join(BASE_DIR, 'thailand_property_full_details.csv')

def extract_content(driver):
    full_content = []

    try:
        title_element = driver.find_element(By.CSS_SELECTOR, '.col-xs-12.col-sm-8 .page-title')
        title_text = title_element.text.strip()
        if title_text:
            full_content.append("--- Listing Title ---")
            full_content.append(title_text)

        location_element = driver.find_element(By.CSS_SELECTOR, '.col-xs-12.col-sm-8 .location')
        loc_text = location_element.text.strip()
        if loc_text:
            full_content.append(f"Location: {loc_text}")
    except Exception:
        pass

    try:
        price_element = driver.find_element(By.CSS_SELECTOR, '.price-title')
        price_text = price_element.text.strip()
        if price_text:
            full_content.append("\n--- Price Information ---")
            full_content.append(price_text)

        try:
            phone_btn = driver.find_element(By.CSS_SELECTOR, '.getAgentBtn')
            driver.execute_script("arguments[0].click();", phone_btn)
            time.sleep(1)
            phone_result = driver.find_element(By.CSS_SELECTOR, '.getAgentBtn').text.strip()
            if phone_result and phone_result != "View Phone":
                full_content.append(f"Phone Number: {phone_result}")
        except Exception:
            pass

    except Exception:
        pass

    try:
        key_features_list = driver.find_elements(By.CSS_SELECTOR, 'ul.key-featured li')
        if key_features_list:
            full_content.append("\n--- Key Features ---")
            features = []
            for li in key_features_list:
                feature_text = li.text.strip().replace('\n', ' ')
                if feature_text:
                    features.append(feature_text)
            if features:
                full_content.append(" | ".join(features))
    except Exception:
        pass

    try:
        desc_container = driver.find_element(By.CSS_SELECTOR, '.col-xs-12.clear-padding .text-description')
        desc_text = desc_container.text.strip()
        if desc_text:
            full_content.append("\n--- Property Details ---")
            clean_desc = desc_text.split("Telephone number:")[0].strip()
            full_content.append(clean_desc)
    except Exception:
        pass

    return "\n".join(full_content)

def main():
    os.makedirs(BASE_DIR, exist_ok=True)

    urls = []

    if os.path.exists(INPUT_CSV):
        with open(INPUT_CSV, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            next(reader, None)
            for row in reader:
                if row and len(row) > 0:
                    urls.append(row[0])
    else:
        print(f"File not found: {INPUT_CSV}")
        return

    if not urls:
        print("No URLs found in CSV.")
        return

    options = uc.ChromeOptions()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-notifications")

    driver = uc.Chrome(options=options, version_main=145)

    print(f"Starting to scrape total {len(urls)} items...")

    with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Post_URL', 'Full_Content'])

        for i, url in enumerate(urls, 1):
            print(f"[{i}/{len(urls)}] Scraping: {url}")
            try:
                driver.get(url)
                time.sleep(3)

                content = extract_content(driver)
                writer.writerow([url, content])

            except Exception as e:
                print(f"Error scraping {url}: {e}")

    driver.quit()
    print(f"\nScraping completed. Saved to: {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

Starting to scrape total 246 items...
[1/246] Scraping: https://www.thailand-property.com/ads/1-bedroom-apartment-for-sale-or-rent-in-avatara_ab96595ae139-d73f-b062-bd3f-b5a8c089
[2/246] Scraping: https://www.thailand-property.com/ads/1-bedroom-apartment-for-sale-or-rent-in-baan-eua-arthorn-nong-hoi-nong-hoi-chiang-mai_6f8783c49455-2790-92e2-4c92-99a8c089
[3/246] Scraping: https://www.thailand-property.com/ads/1-bedroom-condo-for-sale-in-ad-hyatt-condominium-na-kluea-chonburi_86e5d26cb95f-f57e-09c2-bfac-57dcd089
[4/246] Scraping: https://www.thailand-property.com/ads/1-bedroom-condo-for-sale-in-amazon-residence-nong-prue-chonburi_6922e78135e1-e24e-6a62-e9e5-b5a8c089
[5/246] Scraping: https://www.thailand-property.com/ads/1-bedroom-condo-for-sale-in-atlantis-condo-resort-nong-prue-chonburi_105e3be7a3fc-6680-1842-983b-5de5d089
[6/246] Scraping: https://www.thailand-property.com/ads/1-bedroom-condo-for-sale-in-atlantis-condo-resort-nong-prue-chonburi_28074d56e207-fc7e-6372-0a20-6fe5d089
[